# Step 3: Data Preparation
## Horizon-Aware Startup Outcome Prediction

## 3.1 Setup and Raw Data Loading

In [16]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import os

from src.config import (
    FORBIDDEN, FOUNDING_SAFE, FIRST_FUNDING_SAFE, SNAPSHOT_ALL,
    LEAK_REGISTRY, TEMPORAL_SPLIT, TARGET, FUNDING_ROUND_COLS,
    CATEGORICAL_FEATURES, DATE_FEATURES
)
from src.preprocessing import (
    run_cleaning_pipeline, filter_terminal,
    build_h1_features, build_h2_features, build_h3_features,
    temporal_split
)
from src.features import engineer_all_features

SEED = 42
DATA_RAW = '../investments_VC 2.csv'
DATA_OUT = '../data/processed'
os.makedirs(DATA_OUT, exist_ok=True)

# Load raw data
df_raw = pd.read_csv(DATA_RAW, encoding='latin-1')
print(f"Raw CSV loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")

Raw CSV loaded: 54,294 rows, 39 columns


## 3.2 Data Cleaning

All cleaning steps are implemented in `src/preprocessing.py` as reusable functions. The `run_cleaning_pipeline()` function executes them in sequence:

1. Standardise encoding and whitespace
2. Parse `funding_total_usd` (Indian-style commas, dash-as-missing)
3. Remove fully blank rows (4,856 CSV export artefacts)
4. Deduplicate by `permalink` (2 pairs = 4 rows)
5. Parse date columns with out-of-range filtering
6. Flag impossible dates (`first_funding_at < founded_at`)
7. Clean funding round columns to numeric

In [17]:
# Run the full cleaning pipeline
df = run_cleaning_pipeline(df_raw)

=== Data Cleaning Pipeline ===

1. Stripping column/value whitespace...
2. Parsing funding_total_usd...
3. Removing blank rows...
  Removed 4856 blank rows → 49438 remaining
4. Deduplicating by permalink...
  Deduplication: removed 2 duplicate rows by permalink
5. Parsing dates...
  founded_at: 68 out-of-range dates set to NaT
  first_funding_at: 10 out-of-range dates set to NaT
  last_funding_at: 6 out-of-range dates set to NaT
6. Flagging impossible dates...
  Impossible dates (first_funding < founded): 2739 rows flagged
7. Cleaning funding round columns...

=== Cleaning complete: 49436 rows, 44 columns ===


### 3.2.1 Cleaning Summary

Verify cleaning results against EDA audit findings.

In [18]:
# Verify cleaning results
print(f"Cleaned dataset: {len(df):,} rows, {len(df.columns)} columns")
print(f"\nFunding parsed: {df['funding_total_clean'].notna().sum():,} numeric values")
print(f"Impossible date flags: {df['impossible_date_flag'].sum():,} rows")
print(f"\nDate columns parsed:")
for col in ['founded_dt', 'first_funding_dt', 'last_funding_dt']:
    if col in df.columns:
        print(f"  {col}: {df[col].notna().sum():,} valid dates")

# Quick sanity check on funding round columns
print(f"\nFunding round columns (all numeric):")
for col in FUNDING_ROUND_COLS[:5]:
    print(f"  {col}: dtype={df[col].dtype}, non-zero={( df[col] > 0).sum():,}")

Cleaned dataset: 49,436 rows, 44 columns

Funding parsed: 40,905 numeric values
Impossible date flags: 2,739 rows

Date columns parsed:
  founded_dt: 38,484 valid dates
  first_funding_dt: 49,426 valid dates
  last_funding_dt: 49,430 valid dates

Funding round columns (all numeric):
  seed: dtype=float64, non-zero=13,839
  venture: dtype=float64, non-zero=23,275
  equity_crowdfunding: dtype=float64, non-zero=522
  undisclosed: dtype=float64, non-zero=952
  convertible_note: dtype=float64, non-zero=557


### 3.2.2 Impossible Date Decision

2,745 rows have `first_funding_at < founded_at`. Rather than dropping these rows (losing ~7% of date-valid pairs), we **retain them but set their time-to-first-funding to NaN**. This preserves their other features while preventing impossible negative lags from entering the H2 model.

Rationale: Most appear to be imprecise founding dates (e.g., `2010-01-01` as a placeholder). The founding-time features (H1) are still usable; only the H2 lag feature is affected.

In [19]:
# Show impossible date statistics for the terminal subset specifically
terminal_preview = df[df['status'].isin(['acquired', 'closed'])]
n_impossible_terminal = terminal_preview['impossible_date_flag'].sum()
print(f"Impossible dates in terminal subset: {n_impossible_terminal} / {len(terminal_preview)}"
      f" ({n_impossible_terminal/len(terminal_preview)*100:.1f}%)")
print(f"\nDecision: RETAIN rows, set time_to_first_funding_days = NaN for impossible dates")
print("Impact: H1 features unaffected; H2 lag feature will have NaN for these rows (handled by models)")

Impossible dates in terminal subset: 446 / 6295 (7.1%)

Decision: RETAIN rows, set time_to_first_funding_days = NaN for impossible dates
Impact: H1 features unaffected; H2 lag feature will have NaN for these rows (handled by models)


## 3.3 Filter to Terminal Outcomes

Exclude `operating` startups (right-censored) and rows with missing `status`. Keep only `acquired` (target=1) and `closed` (target=0).

In [20]:
# Filter to terminal outcomes
df_term = filter_terminal(df)
print(f"\nClass balance: {df_term['target'].mean():.3f} acquired rate")

  Terminal subset: 6295 rows (acquired=3692, closed=2603)

Class balance: 0.586 acquired rate


## 3.4 Feature Engineering

All feature engineering is implemented in `src/features.py`. Features are organised by horizon level:

**H1 (Founding-time):**
- `founding_year`, `founding_quarter` — derived from `founded_at`
- `num_categories` — count of pipe-separated categories
- `primary_category` — first category extracted
- `market_clean` — stripped and lowered market
- `is_usa` — binary USA flag
- `has_state` — whether state_code is non-null

**H2 (First-funding):**
- `time_to_first_funding_days` — days from founding to first funding (NaN for impossible dates)

**H3 (Snapshot):**
- `num_funding_types` — count of non-zero funding type columns
- `max_round_reached` — ordinal encoding of highest round (A=1, ..., H=8)
- `funding_total_clean` — parsed total funding (already created in cleaning)

In [21]:
# Apply all feature engineering
df_term = engineer_all_features(df_term)

# Show sample of engineered features
eng_cols = ['founding_year', 'founding_quarter', 'num_categories', 'primary_category',
            'market_clean', 'is_usa', 'has_state', 'time_to_first_funding_days',
            'num_funding_types', 'max_round_reached']
print("\nSample of engineered features:")
df_term[eng_cols].head(10)

=== Feature Engineering ===
1. Founding date features (year, quarter)...
2. Category features (primary_category, num_categories)...
3. Market cleaning...
4. Geography flags (is_usa, has_state)...
5. Time-to-first-funding (H2)...
6. Num funding types (H3)...
7. Max round reached (H3)...

=== Feature engineering complete: 10 new columns added ===

Sample of engineered features:


,founding_year,founding_quarter,num_categories,primary_category,market_clean,is_usa,has_state,time_to_first_funding_days,num_funding_types,max_round_reached
0,2012,2,4,Entertainment,news,1,1,29.0,1,0
1,<NA>,<NA>,1,Advertising,advertising,0,0,NaN,1,0
2,2009,1,3,Marketplaces,marketplaces,0,0,134.0,1,0
3,2010,3,1,Curated Web,curated web,1,1,NaN,3,1
4,2011,3,1,Analytics,analytics,1,1,47.0,2,0
5,2009,2,1,Curated Web,curated web,1,1,0.0,1,0
6,2012,1,1,E-Commerce,e-commerce,1,1,95.0,0,0
7,1990,1,1,Software,software,1,1,4488.0,2,1
8,2006,1,2,Cars,curated web,0,0,767.0,0,0
9,2006,2,4,Lifestyle,lifestyle,1,1,803.0,0,0


In [22]:
# Verify feature engineering statistics
print("Feature engineering summary:")
print(f"  founding_year: {df_term['founding_year'].notna().sum()} valid ({df_term['founding_year'].isna().sum()} missing)")
print(f"  time_to_first_funding_days: {df_term['time_to_first_funding_days'].notna().sum()} valid "
      f"(median={df_term['time_to_first_funding_days'].median():.0f} days)")
print(f"  num_categories: mean={df_term['num_categories'].mean():.1f}, max={df_term['num_categories'].max()}")
print(f"  primary_category: {df_term['primary_category'].nunique()} unique values")
print(f"  market_clean: {df_term['market_clean'].nunique()} unique values")
print(f"  is_usa: {df_term['is_usa'].mean():.1%} are USA-based")
print(f"  has_state: {df_term['has_state'].mean():.1%} have state_code")
print(f"  num_funding_types: mean={df_term['num_funding_types'].mean():.1f}")
print(f"  max_round_reached: distribution:")
print(df_term['max_round_reached'].value_counts().sort_index())

Feature engineering summary:
  founding_year: 4967 valid (1328 missing)
  time_to_first_funding_days: 4520 valid (median=578 days)
  num_categories: mean=2.2, max=15
  primary_category: 468 unique values
  market_clean: 407 unique values
  is_usa: 67.5% are USA-based
  has_state: 70.2% have state_code
  num_funding_types: mean=1.7
  max_round_reached: distribution:
max_round_reached
0    3677
1     963
2     787
3     487
4     246
5     101
6      30
7       4
Name: count, dtype: int64


## 3.5 Build Three Horizon Tables

Construct three separate feature sets from the cleaned data, each enforcing strict temporal boundaries per `src/config.py` registries. The horizon table builders in `src/preprocessing.py` ensure:

- No FORBIDDEN column enters any horizon
- No derive-then-drop column (raw date strings) enters any horizon
- No snapshot-only column enters H1 or H2
- Engineered features are placed at the correct horizon level

### 3.5.1 H1: Founding Dataset

Features from `FOUNDING_SAFE` only. Raw date columns (`founded_at`) are used to derive features (year, quarter) and then dropped -- they must NOT enter the model directly.

In [23]:
# Build H1 feature matrix
X_h1 = build_h1_features(df_term)
y = df_term['target']

print(f"H1 (Founding) feature matrix: {X_h1.shape}")
print(f"Columns: {list(X_h1.columns)}")
print(f"\nMissingness in H1:")
miss_h1 = X_h1.isna().sum()
print(miss_h1[miss_h1 > 0] if (miss_h1 > 0).any() else "  No missing values")

H1 (Founding) feature matrix: (6295, 11)
Columns: ['country_code', 'state_code', 'region', 'city', 'founding_year', 'founding_quarter', 'num_categories', 'primary_category', 'market_clean', 'is_usa', 'has_state']

Missingness in H1:
country_code         628
state_code          1878
region               628
city                 691
founding_year       1328
founding_quarter    1328
primary_category     229
market_clean         231
dtype: int64


### 3.5.2 H2: First-Funding Dataset

H1 features plus `time_to_first_funding_days`.

In [24]:
# Build H2 feature matrix
X_h2 = build_h2_features(df_term)

print(f"H2 (First-Funding) feature matrix: {X_h2.shape}")
print(f"Columns: {list(X_h2.columns)}")
print(f"New vs H1: {set(X_h2.columns) - set(X_h1.columns)}")
print(f"\ntime_to_first_funding_days stats:")
print(X_h2['time_to_first_funding_days'].describe())

H2 (First-Funding) feature matrix: (6295, 12)
Columns: ['country_code', 'state_code', 'region', 'city', 'founding_year', 'founding_quarter', 'num_categories', 'primary_category', 'market_clean', 'is_usa', 'has_state', 'time_to_first_funding_days']
New vs H1: {'time_to_first_funding_days'}

time_to_first_funding_days stats:
count     4520.000000
mean      1303.286947
std       2065.278411
min          0.000000
25%        176.000000
50%        578.000000
75%       1646.250000
max      37466.000000
Name: time_to_first_funding_days, dtype: float64


### 3.5.3 H3: Snapshot Dataset

H2 features plus all lifetime funding aggregates and derived snapshot features.

In [25]:
# Build H3 feature matrix
X_h3 = build_h3_features(df_term)

print(f"H3 (Snapshot) feature matrix: {X_h3.shape}")
print(f"Columns: {list(X_h3.columns)}")
print(f"New vs H2: {set(X_h3.columns) - set(X_h2.columns)}")
print(f"\nH3 feature count breakdown:")
print(f"  From H1: {len(X_h1.columns)} features")
print(f"  From H2 (additional): {len(X_h2.columns) - len(X_h1.columns)} features")
print(f"  From H3 (additional): {len(X_h3.columns) - len(X_h2.columns)} features")
print(f"  Total H3: {len(X_h3.columns)} features")

H3 (Snapshot) feature matrix: (6295, 35)
Columns: ['country_code', 'state_code', 'region', 'city', 'founding_year', 'founding_quarter', 'num_categories', 'primary_category', 'market_clean', 'is_usa', 'has_state', 'time_to_first_funding_days', 'funding_rounds', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H', 'num_funding_types', 'max_round_reached', 'funding_total_clean']
New vs H2: {'undisclosed', 'round_H', 'funding_total_clean', 'secondary_market', 'max_round_reached', 'grant', 'round_E', 'round_B', 'round_D', 'round_C', 'round_A', 'num_funding_types', 'venture', 'angel', 'funding_rounds', 'seed', 'debt_financing', 'private_equity', 'equity_crowdfunding', 'round_F', 'convertible_note', 'product_crowdfunding', 'round_G'}

H3 feature count breakdown:
  From H1: 11 features
 

## 3.6 Horizon Validation

Run automated leakage checks before proceeding. These checks enforce the invariants from `src/config.py`.

In [26]:
# ── Horizon integrity checks ──────────────────────────────────────────
print("=== Horizon Leakage Validation ===\n")

# 1. No FORBIDDEN column in any horizon
for name, X in [("H1", X_h1), ("H2", X_h2), ("H3", X_h3)]:
    forbidden_present = set(X.columns) & set(FORBIDDEN)
    assert len(forbidden_present) == 0, f"{name} contains FORBIDDEN columns: {forbidden_present}"
    print(f"[PASS] {name}: No FORBIDDEN columns")

# 2. No raw date strings in any horizon
raw_dates = ['founded_at', 'first_funding_at', 'last_funding_at']
for name, X in [("H1", X_h1), ("H2", X_h2), ("H3", X_h3)]:
    dates_present = set(X.columns) & set(raw_dates)
    assert len(dates_present) == 0, f"{name} contains raw date columns: {dates_present}"
    print(f"[PASS] {name}: No raw date strings")

# 3. No snapshot-only columns in H1
snapshot_only_cols = set(SNAPSHOT_ALL) - set(FOUNDING_SAFE)
# Also include H3 engineered features
h3_engineered = {'num_funding_types', 'max_round_reached', 'funding_total_clean',
                 'time_to_first_funding_days'}
snapshot_leak_in_h1 = set(X_h1.columns) & (snapshot_only_cols | h3_engineered)
assert len(snapshot_leak_in_h1) == 0, f"H1 contains snapshot columns: {snapshot_leak_in_h1}"
print(f"[PASS] H1: No snapshot-only columns leaked")

# 4. No snapshot-only columns in H2 (except time_to_first_funding_days)
h2_allowed_extras = {'time_to_first_funding_days'}
snapshot_leak_in_h2 = (set(X_h2.columns) & (snapshot_only_cols | h3_engineered)) - h2_allowed_extras
assert len(snapshot_leak_in_h2) == 0, f"H2 contains snapshot columns: {snapshot_leak_in_h2}"
print(f"[PASS] H2: No snapshot-only columns leaked (time_to_first_funding_days correctly included)")

# 5. Target not in any feature matrix
for name, X in [("H1", X_h1), ("H2", X_h2), ("H3", X_h3)]:
    assert 'target' not in X.columns, f"{name} contains target column"
    assert 'status' not in X.columns, f"{name} contains status column"
    print(f"[PASS] {name}: No target/status leakage")

# 6. All horizons have same number of rows
assert len(X_h1) == len(X_h2) == len(X_h3) == len(y), \
    f"Row count mismatch: H1={len(X_h1)}, H2={len(X_h2)}, H3={len(X_h3)}, y={len(y)}"
print(f"\n[PASS] All horizons: {len(y)} rows each")

print("\n=== All horizon leakage checks passed ===")

=== Horizon Leakage Validation ===

[PASS] H1: No FORBIDDEN columns
[PASS] H2: No FORBIDDEN columns
[PASS] H3: No FORBIDDEN columns
[PASS] H1: No raw date strings
[PASS] H2: No raw date strings
[PASS] H3: No raw date strings
[PASS] H1: No snapshot-only columns leaked
[PASS] H2: No snapshot-only columns leaked (time_to_first_funding_days correctly included)
[PASS] H1: No target/status leakage
[PASS] H2: No target/status leakage
[PASS] H3: No target/status leakage

[PASS] All horizons: 6295 rows each

=== All horizon leakage checks passed ===


## 3.7 Temporal Split

Split by `first_funding_year`:
- **Train:** first_funding_year <= 2008
- **Validation:** 2009-2010
- **Test:** >= 2011

Rows with missing first_funding dates fall back to founding_year. Rows with no temporal info are assigned to training.

In [27]:
# Temporal split for all three horizons
print("=== Temporal Split ===\n")

splits = {}
for name, X in [("H1", X_h1), ("H2", X_h2), ("H3", X_h3)]:
    print(f"\n--- {name} ---")
    X_train, X_val, X_test, y_train, y_val, y_test = temporal_split(df_term, X, y)
    splits[name] = {
        'X_train': X_train, 'X_val': X_val, 'X_test': X_test,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    }
    # Verify no data leakage in split
    total = len(X_train) + len(X_val) + len(X_test)
    assert total == len(y), f"{name}: split sizes don't sum to total ({total} != {len(y)})"
    print(f"  [PASS] Split sizes sum to {total}")

print("\n=== All temporal splits complete ===")

=== Temporal Split ===


--- H1 ---
  1 rows with missing split year assigned to train
  Temporal split sizes: train=3218, val=1625, test=1452
  Class balance: train=0.656, val=0.503, test=0.526
  [PASS] Split sizes sum to 6295

--- H2 ---
  1 rows with missing split year assigned to train
  Temporal split sizes: train=3218, val=1625, test=1452
  Class balance: train=0.656, val=0.503, test=0.526
  [PASS] Split sizes sum to 6295

--- H3 ---
  1 rows with missing split year assigned to train
  Temporal split sizes: train=3218, val=1625, test=1452
  Class balance: train=0.656, val=0.503, test=0.526
  [PASS] Split sizes sum to 6295

=== All temporal splits complete ===


## 3.8 Encoding Strategy

Different models require different encoding of categorical features:

- **CatBoost:** Handles categoricals natively via `cat_features` parameter. No encoding needed.
- **HistGradientBoostingClassifier:** Requires `OrdinalEncoder` for categoricals. Handles NaN natively.
- **LogisticRegression:** High-cardinality fields (market, primary_category) need frequency encoding. StandardScaler for numerics.
- **TabM:** Follows official repo preprocessing (numeric + categorical handled separately).

Encoding is applied **per-model at training time** in Notebook 04, not here. This avoids data leakage from fitting encoders on the full dataset before splitting. We save the raw (unencoded) horizon splits below.

In [28]:
# Identify categorical vs numeric columns in each horizon
for name, X in [("H1", X_h1), ("H2", X_h2), ("H3", X_h3)]:
    cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    print(f"\n{name}:")
    print(f"  Categorical ({len(cat_cols)}): {cat_cols}")
    print(f"  Numeric ({len(num_cols)}): {num_cols}")

print("\nNote: Encoding will be applied per-model in Notebook 04.")


H1:
  Categorical (6): ['country_code', 'state_code', 'region', 'city', 'primary_category', 'market_clean']
  Numeric (5): ['founding_year', 'founding_quarter', 'num_categories', 'is_usa', 'has_state']

H2:
  Categorical (6): ['country_code', 'state_code', 'region', 'city', 'primary_category', 'market_clean']
  Numeric (6): ['founding_year', 'founding_quarter', 'num_categories', 'is_usa', 'has_state', 'time_to_first_funding_days']

H3:
  Categorical (6): ['country_code', 'state_code', 'region', 'city', 'primary_category', 'market_clean']
  Numeric (29): ['founding_year', 'founding_quarter', 'num_categories', 'is_usa', 'has_state', 'time_to_first_funding_days', 'funding_rounds', 'seed', 'venture', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'private_equity', 'secondary_market', 'product_crowdfunding', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'round_G', 'round_H', 'num_funding_types', 'max_round_reached', 'fundin

## 3.9 Save Processed Datasets

Save all horizon splits to `data/processed/` as CSV files. Each horizon gets train/val/test X and y files.

In [29]:
# Save all horizon datasets
print("=== Saving Processed Datasets ===\n")

for horizon_name in ["H1", "H2", "H3"]:
    s = splits[horizon_name]
    for split_name in ["train", "val", "test"]:
        X_key = f"X_{split_name}"
        y_key = f"y_{split_name}"
        X_path = os.path.join(DATA_OUT, f"{horizon_name}_{split_name}_X.csv")
        y_path = os.path.join(DATA_OUT, f"{horizon_name}_{split_name}_y.csv")
        s[X_key].to_csv(X_path, index=False)
        s[y_key].to_csv(y_path, index=False, header=True)
        print(f"  Saved {horizon_name}_{split_name}: X={s[X_key].shape}, y={s[y_key].shape}")

# Also save the full cleaned terminal dataset for reference
df_term.to_csv(os.path.join(DATA_OUT, "terminal_cleaned.csv"), index=False)
print(f"\n  Saved terminal_cleaned.csv: {df_term.shape}")

print(f"\nAll files saved to {DATA_OUT}/")
print(f"Total files: {len(os.listdir(DATA_OUT))}")

=== Saving Processed Datasets ===

  Saved H1_train: X=(3218, 11), y=(3218,)
  Saved H1_val: X=(1625, 11), y=(1625,)
  Saved H1_test: X=(1452, 11), y=(1452,)
  Saved H2_train: X=(3218, 12), y=(3218,)
  Saved H2_val: X=(1625, 12), y=(1625,)
  Saved H2_test: X=(1452, 12), y=(1452,)
  Saved H3_train: X=(3218, 35), y=(3218,)
  Saved H3_val: X=(1625, 35), y=(1625,)
  Saved H3_test: X=(1452, 35), y=(1452,)

  Saved terminal_cleaned.csv: (6295, 55)

All files saved to ../data/processed/
Total files: 19


## 3.10 Data Preparation Summary

In [30]:
# Final summary
print("=" * 60)
print("DATA PREPARATION SUMMARY")
print("=" * 60)

print(f"\nRaw data:      {df_raw.shape[0]:>6,} rows x {df_raw.shape[1]} columns")
print(f"After cleaning: {len(df):>6,} rows x {len(df.columns)} columns")
print(f"Terminal subset: {len(df_term):>6,} rows (acquired={df_term['target'].sum():,}, closed={(1-df_term['target']).sum():,.0f})")

print(f"\nHorizon feature matrices:")
print(f"  H1 (Founding):       {X_h1.shape[1]:>2} features")
print(f"  H2 (First-Funding):  {X_h2.shape[1]:>2} features")
print(f"  H3 (Snapshot):       {X_h3.shape[1]:>2} features")

print(f"\nTemporal split (same for all horizons):")
s = splits["H1"]
print(f"  Train (<=2008):    {len(s['X_train']):>5,} rows, acquired rate={s['y_train'].mean():.3f}")
print(f"  Val (2009-2010):   {len(s['X_val']):>5,} rows, acquired rate={s['y_val'].mean():.3f}")
print(f"  Test (>=2011):     {len(s['X_test']):>5,} rows, acquired rate={s['y_test'].mean():.3f}")

print(f"\nKey decisions:")
print(f"  - Impossible dates: retained, time_to_first_funding set to NaN")
print(f"  - Encoding: deferred to modelling notebook (per-model)")
print(f"  - Temporal split: by first_funding_year (fallback: founding_year)")
print(f"\n  Processed files saved to data/processed/")

DATA PREPARATION SUMMARY

Raw data:      54,294 rows x 39 columns
After cleaning: 49,436 rows x 44 columns
Terminal subset:  6,295 rows (acquired=3,692, closed=2,603)

Horizon feature matrices:
  H1 (Founding):       11 features
  H2 (First-Funding):  12 features
  H3 (Snapshot):       35 features

Temporal split (same for all horizons):
  Train (<=2008):    3,218 rows, acquired rate=0.656
  Val (2009-2010):   1,625 rows, acquired rate=0.503
  Test (>=2011):     1,452 rows, acquired rate=0.526

Key decisions:
  - Impossible dates: retained, time_to_first_funding set to NaN
  - Encoding: deferred to modelling notebook (per-model)
  - Temporal split: by first_funding_year (fallback: founding_year)

  Processed files saved to data/processed/
